<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Solution: Next item from browsing sessions

**[Next item from browsing sessions](https://learning.nextia-ai.com/courses/recommendations/m10/next-item-from-browsing-sessions/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 10, case study 2 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** compare popularity, co-visitation, a last-item (Markov) model and ALS on real browsing sessions of anonymous visitors, measure clicks, carts and orders separately, and decide what a product page should suggest.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in about 5 minutes |
| **Needs** | An internet connection for the setup cells. No account. Not the course project. |
| **You should know** | Recall@k and segments, from [Evaluate by segment](https://learning.nextia-ai.com/courses/recommendations/m03/evaluate-by-segment/). Session suggestions, from [Sessions, sequences and LLMs, briefly](https://learning.nextia-ai.com/courses/recommendations/m07/sessions-sequences-and-llms-briefly/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1, matplotlib 3.11.2 and implicit 0.7.3, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*). Colab and Kaggle were not run. |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/next-item-from-browsing-sessions/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M10-L02-next-item-from-browsing-sessions/next-item-from-browsing-sessions-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1 and implicit 0.7.3 (ALS), and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2 implicit==0.7.3
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy', 'implicit']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It makes a folder `browsing-sessions` with a `data` folder in it. It downloads `otto-sample.csv.gz` (20 MB) from the course's labs repository and checks the file's checksum (SHA-256). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready:` and the file with its size. If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `browsing-sessions` (in Colab: the **Files** panel on the left), then run the cell again.

The data is a sample of the **OTTO Recommender Systems Dataset** by Philipp Normann, Sophie Baumeister and Timo Wilm ([github.com/otto-de/recsys-dataset](https://github.com/otto-de/recsys-dataset)), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The course kept every 5th visit of the test week, wrote one row per event and changed the timestamps from milliseconds to seconds; the [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M10-L02-next-item-from-browsing-sessions/dataset.md) lists the details. It is real data from a German online shop. The IDs are anonymised; do not try to identify anyone.

In [ ]:
# Setup: download the data into browsing-sessions/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/M10-L02-next-item-from-browsing-sessions/"
FILES = {
    "otto-sample.csv.gz": "c8d74048524643a0ebf0abd7a3be10792a9895648b21332727990e6588c80a91",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into another project. If the cell ran before, start from the parent.
if Path(".nextia-notebook").exists():
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("browsing-sessions").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder browsing-sessions and run again.")

os.chdir(WORK)
print("Ready:", ", ".join(f"data/{n} ({Path('data', n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the case study's tools. Read the comments if you like; the lesson page explains each one.

- `read(path)` loads the events. Products get small numbers (`item`) so that they fit in a matrix.
- `Split(events, day)` is the time split: everything known before the day (`history`), and the visits that start on the day, each cut at a random point (`seen` events visible, the rest to predict).
- `session_weights`, `session_matrix`, `top_k`, `fill` and `combine` turn scores into lists of 20 products.
- `evaluate(split, lists)` measures the lists: recall@20 for the next click, for carts and for orders, each with a 95% interval; OTTO's weighted score; MRR@20 of the next click; coverage; and the score by segment (the length of the visit so far).
- `expect(...)` and `expect_hash(...)` check your answers without showing them.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, lists and measures (the same code as the lesson page).
import hashlib, json, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

K = 20                                      # the length of every list


TYPES = ("clicks", "carts", "orders")


DAY0 = 1661724000                           # Monday 29 August 2022, 00:00 in Germany (UTC+2)


DAYS = ["Mon 29 Aug", "Tue 30 Aug", "Wed 31 Aug", "Thu 1 Sep", "Fri 2 Sep", "Sat 3 Sep", "Sun 4 Sep"]


TYPE_WEIGHT = np.array([1.0, 6.0, 3.0])     # a cart says more than a click, an order more than a click


SEGMENTS = ("1 event", "2 to 4 events", "5 or more")


def read(path):
    """One row per event, in the file's order (by session, then time). type: 0 click, 1 cart, 2 order."""
    ev = pd.read_csv(path, dtype={"session": "int32", "aid": "int32", "ts": "int32", "type": "category"})
    ev["type"] = ev["type"].cat.rename_categories({"clicks": 0, "carts": 1, "orders": 2}).astype("int8")
    ev["day"] = ((ev["ts"] - DAY0) // 86400).astype("int8")
    ev["item"], items = pd.factorize(ev["aid"], sort=True)    # products numbered 0, 1, 2 ...
    ev["item"] = ev["item"].astype("int32")
    return ev, np.asarray(items)


class Split:
    """The time split for one day.

    history:  every event before the day starts (all the models may learn from).
    sessions: the visits that start on the day. Each one is cut at a random point: the visitor has done the
              first `seen` events, and we predict the rest: the next click, and every product carted or ordered.
    """

    def __init__(self, ev, day, seed=37):
        self.day, self.start = day, DAY0 + day * 86400
        self.n_items = int(ev["item"].max()) + 1
        self.history = ev[ev["ts"] < self.start]
        first = ev.groupby("session", sort=False)["ts"].transform("min")
        visits = ev[(first >= self.start) & (first < self.start + 86400)]
        self.sessions, starts, lengths = np.unique(visits["session"].to_numpy(), return_index=True,
                                                   return_counts=True)
        rng = np.random.default_rng(seed)
        self.seen = 1 + (rng.random(len(lengths)) * (lengths - 1)).astype(int)   # 1 to length - 1 events
        pos = np.arange(len(visits)) - np.repeat(starts, lengths)
        is_seen = pos < np.repeat(self.seen, lengths)
        self.visible = visits[is_seen]
        future = visits[~is_seen].assign(row=np.repeat(np.arange(len(lengths)), lengths)[~is_seen])
        n = len(self.sessions)
        clicks = future[future["type"] == 0].drop_duplicates("row")
        self.next_click = np.full(n, -1)
        self.next_click[clicks["row"].to_numpy()] = clicks["item"].to_numpy()
        self.truth = {}
        for t, name in ((1, "carts"), (2, "orders")):
            f = future[future["type"] == t]
            self.truth[name] = f.groupby("row")["item"].agg(lambda s: set(s.tolist())).to_dict()
        self.segment = np.where(self.seen == 1, SEGMENTS[0], np.where(self.seen <= 4, SEGMENTS[1], SEGMENTS[2]))
        self.rows = np.repeat(np.arange(n), self.seen)          # session row of each visible event
        self.seen_sets = [set() for _ in range(n)]
        for r, i in zip(self.rows, self.visible["item"].to_numpy()):
            self.seen_sets[r].add(i)

    def __len__(self):
        return len(self.sessions)


def fill(lists, fallback):
    """Fill the empty places (-1) of each list with the fallback list, without repeats."""
    out = lists.copy()
    for row in out:
        have = set(row[row >= 0].tolist())
        extra = [p for p in fallback if p not in have]
        empty = np.flatnonzero(row < 0)
        row[empty] = extra[:len(empty)]
    return out


def top_k(scores, k=K):
    """The k best columns of each row of a sparse score matrix (-1 where a row has fewer)."""
    scores = sp.csr_matrix(scores)
    out = np.full((scores.shape[0], k), -1)
    for r in range(scores.shape[0]):
        a, b = scores.indptr[r], scores.indptr[r + 1]
        cols, vals = scores.indices[a:b], scores.data[a:b]
        keep = vals > 0
        cols, vals = cols[keep], vals[keep]
        best = np.lexsort((cols, -vals))[:k]                  # highest score first; ties: the lower number
        out[r, :len(best)] = cols[best]
    return out


def session_weights(split):
    """One weight per visible event: type weight x recency (0.1 for the first event of a visit, 1 for the last)."""
    seen = split.seen
    pos = np.arange(len(split.rows)) - np.repeat(np.cumsum(seen) - seen, seen)
    recency = np.where(np.repeat(seen, seen) > 1, 0.1 + 0.9 * pos / np.maximum(np.repeat(seen, seen) - 1, 1), 1.0)
    return TYPE_WEIGHT[split.visible["type"].to_numpy()] * recency


def session_matrix(split, weights):
    """Sessions x products: the summed weights of each session's visible events."""
    return sp.csr_matrix((weights, (split.rows, split.visible["item"].to_numpy())),
                         shape=(len(split), split.n_items))


def combine(first, then):
    """The products of `first` (in order), then those of `then` that are not in it yet."""
    out = np.full_like(first, -1)
    for r in range(len(first)):
        seen, row = set(), []
        for p in list(first[r]) + list(then[r]):
            if p >= 0 and p not in seen:
                seen.add(p)
                row.append(p)
                if len(row) == K:
                    break
        out[r, :len(row)] = row
    return out


def hits(split, lists):
    """Per visit: recall@20 of the next click, of the carts and of the orders (NaN where the visit has none)."""
    n = len(split)
    out = {t: np.full(n, np.nan) for t in TYPES}
    has = split.next_click >= 0
    out["clicks"][has] = (lists[has] == split.next_click[has, None]).any(axis=1)
    for t in ("carts", "orders"):
        for r, truth in split.truth[t].items():
            out[t][r] = len(truth & set(lists[r].tolist())) / min(K, len(truth))
    return out


def score(per_visit):
    """OTTO's weighted score: 0.1 x clicks + 0.3 x carts + 0.6 x orders (each the mean over visits)."""
    return 0.1 * np.nanmean(per_visit["clicks"]) + 0.3 * np.nanmean(per_visit["carts"]) + \
        0.6 * np.nanmean(per_visit["orders"])


def bootstrap(per_visit, rounds=200, seed=37):
    """95% intervals over visits for the three recalls and the score (the visits resampled together)."""
    rng = np.random.default_rng(seed)
    n = len(per_visit["clicks"])
    draws = {t: [] for t in (*TYPES, "score")}
    for _ in range(rounds):
        idx = rng.integers(0, n, n)
        sample = {t: per_visit[t][idx] for t in TYPES}
        for t in TYPES:
            draws[t].append(np.nanmean(sample[t]))
        draws["score"].append(score(sample))
    return {t: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for t, v in draws.items()}


def evaluate(split, lists):
    """The three recalls with intervals, the score, MRR@20 of the next click, coverage, re-shown share and
    recall by segment. Also returns the per-visit values (for paired comparisons)."""
    per = hits(split, lists)
    ci = bootstrap(per)
    out = {t: [float(np.nanmean(per[t])), *ci[t]] for t in TYPES}
    out["score"] = [float(score(per)), *ci["score"]]
    has = split.next_click >= 0
    place = (lists[has] == split.next_click[has, None])
    out["mrr"] = float(np.where(place.any(axis=1), 1 / (place.argmax(axis=1) + 1), 0).mean())
    out["coverage"] = len(set(lists[lists >= 0].tolist())) / split.n_items
    out["reshown"] = float(np.mean([np.mean([p in s for p in row if p >= 0]) for row, s in
                                    zip(lists, split.seen_sets)]))
    new_click = has & np.array([c not in s for c, s in zip(split.next_click, split.seen_sets)])
    out["clicks_new"] = float(np.nanmean(per["clicks"][new_click]))
    out["segments"] = {g: {t: float(np.nanmean(per[t][split.segment == g])) for t in TYPES} |
                       {"score": float(score({t: per[t][split.segment == g] for t in TYPES}))} for g in SEGMENTS}
    return out, per


def timed(f, *args, **kwargs):
    start = time.perf_counter()
    result = f(*args, **kwargs)
    return result, time.perf_counter() - start


def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A large online shop wants a block of 20 products on its product pages for visitors who are **not logged in**. The shop knows nothing about these visitors except what they did in this visit: the products they looked at (clicks), put in the basket (carts) and ordered. Lena, the product manager, asks: "What should the block show, and how much better than our bestsellers is it?" Jonas, the engineer, asks: "What does it cost to run?" Lena and Jonas are made up, and they do not work for OTTO. The data is real.

Your job: compare popularity, the visit's own products, co-visitation, a last-item (Markov) model and ALS on an honest time split, measure recall@20 for clicks, carts and orders separately, and say what the block should show. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/next-item-from-browsing-sessions/#the-situation) tells the full story.

## 2. The data

Run the next cell. It reads the events and counts them by type and by day. The days are German days (the data is in UTC).

> **Predict.** A visit in this data has at least 2 events. What share of visits have exactly 2: about 10%, about 25%, or about 40%?

> **Check.** You should see 2,763,918 events in 334,361 visits, on 491,129 different products.

In [ ]:
events, aids = read("data/otto-sample.csv.gz")
print(f"{len(events):,} events in {events.session.nunique():,} visits, on {events['item'].nunique():,} products")
print(events["type"].map(dict(enumerate(TYPES))).value_counts().to_string())
lengths = events.groupby("session").size()
print(f"events per visit: median {lengths.median():.0f}, mean {lengths.mean():.1f}; "
      f"visits with exactly 2 events: {(lengths == 2).mean():.1%}")
print(pd.Series(events.groupby("day").size().to_numpy(), index=DAYS).to_string())

**What this tells us.** Most events are clicks; carts are about one event in eleven and orders one in forty. Half of the visits have 4 events or fewer. The catalogue is huge: 491,129 products in a fifth of one week.

## 3. The time split and the cut

Run the next cell. It makes the test day: **Saturday 3 September**. The models learn from every event before Saturday. The test visits are the visits that start on Saturday. Each visit is cut at a random point: the visitor has done the first `seen` events, and we predict the rest (the next click, and every product carted or ordered later in the visit). The segments are the length of the visit so far: 1 event, 2 to 4 events, or 5 or more.

> **Check.** You should see 41,273 visits: 18,689 with 1 event seen, 13,783 with 2 to 4 and 8,801 with 5 or more.

In [ ]:
test = Split(events, day=5)
print(f"{len(test):,} visits start on {DAYS[5]}; history: {len(test.history):,} events before it")
print(pd.Series(test.segment).value_counts().reindex(SEGMENTS).to_string())
print(f"visits with a next click: {(test.next_click >= 0).sum():,}, with carts later: {len(test.truth['carts']):,}, "
      f"with orders later: {len(test.truth['orders']):,}")

> **Your turn.** What share of the products carted in the rest of a visit had the visitor **already seen** in the visible part? Use `test.truth["carts"]` (visit row → set of products carted later) and `test.seen_sets` (one set of seen products per visit row). Count each (visit, product) pair once. Store the share in `share_seen_carts`.

In [ ]:
pairs = [(product in test.seen_sets[row]) for row, carted in test.truth["carts"].items() for product in carted]
share_seen_carts = sum(pairs) / len(pairs)
print(round(share_seen_carts, 3))

In [ ]:
expect('share_seen_carts', rounded(share_seen_carts), 0.326, tolerance=0.002)

**What this tells us.** Visitors often come back to a product they already looked at before they put it in the basket: 32.6% of later carts, and 61.4% of later orders. For the next click it is 32.2%. Keep these numbers in mind: they decide the case study.

## 4. The baselines: popular, and the visit so far

Run the next cell. `popular` ranks products by the number of visits that touched them in the 24 hours before Saturday. Everyone gets the same list. `session_items` shows the visit's own products: carts count 6 times a click, orders 3 times, and recent events count more. A visit with fewer than 20 products is filled with popular products.

> **Predict.** Which will be better for orders: the 20 most popular products, or the products of the visit itself?

In [ ]:
def popular(split, hours=24):
    """Products ranked by the number of visits that touched them in the last `hours` before the day."""
    h = split.history
    recent = h[h["ts"] >= split.start - hours * 3600].drop_duplicates(["session", "item"])
    counts = np.bincount(recent["item"], minlength=split.n_items)
    return np.argsort(-counts, kind="stable")[:100]          # 100, so that fill() never runs short


def session_items(split):
    """The products the visitor has already seen, carted or ordered in this visit, by weight."""
    return top_k(session_matrix(split, session_weights(split)))


results = {}
pop = popular(test)
blank = np.full((len(test), K), -1)
lists = {"popular": fill(blank, pop), "visit so far": fill(session_items(test), pop)}
for name in lists:
    results[name], _ = evaluate(test, lists[name])
    print(f"{name:14} clicks {results[name]['clicks'][0]:.3f}  carts {results[name]['carts'][0]:.3f}  "
          f"orders {results[name]['orders'][0]:.3f}  score {results[name]['score'][0]:.3f}")

> **Check.** You should see a score of 0.014 for popular and 0.568 for the visit so far.

The **score** is OTTO's weighted score: 0.1 × clicks + 0.3 × carts + 0.6 × orders. Orders weigh most, because they are what the shop sells.

## 5. Co-visitation

"Visitors who looked at this also looked at that." Count, for each pair of products, the past visits in which both were seen. First on three tiny visits that you can check by hand:

In [ ]:
# Three past visits, each a list of products in the order they were looked at.
tiny = {"visit 1": ["tent", "sleeping mat", "lamp"],
        "visit 2": ["tent", "sleeping mat"],
        "visit 3": ["tent", "lamp", "stove"]}
counts = {}
for products in tiny.values():
    for a in set(products):
        for b in set(products):
            if a != b:
                counts[(a, b)] = counts.get((a, b), 0) + 1     # once per visit
print("Visitors who looked at the tent also looked at:")
for (a, b), n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0][1])):   # most visits first, then A to Z
    if a == "tent":
        print(f"  {b:13} in {n} of 3 visits")

> **Check.** You should see `lamp` in 2 of 3 visits, `sleeping mat` in 2 of 3 and `stove` in 1 of 3. Count them yourself in the three lists.

Run the next cell. It does the same on every visit before Saturday: pairs of products seen in the same visit within 24 hours, counted once per visit, in the last 30 events of each visit. It keeps the 20 best neighbours of each product. Then it scores each product for a test visit by adding its counts with the products of the visit (recent events and carts count more). This takes about 20 seconds.

In [ ]:
def covisitation(split, hours=24, last=30, neighbours=20):
    """'Visitors who looked at this also looked at that': for each product, the products seen in the same visit
    within `hours`, counted once per visit, in the last `last` events of each past visit. Keeps the top
    `neighbours` of each product. (pandas is too slow for this; the loop pairs each event with the next ones.)"""
    h = split.history.groupby("session", sort=False).tail(last)
    s = h["session"].to_numpy()
    i = h["item"].to_numpy().astype(np.int64)
    t = h["ts"].to_numpy()
    keys = []
    for d in range(1, last):                    # pair each event with the events d places later
        same = (s[d:] == s[:-d]) & (t[d:] - t[:-d] < hours * 3600) & (i[d:] != i[:-d])
        a, b, v = i[:-d][same], i[d:][same], s[d:][same].astype(np.int64)
        keys += [(v << 42) | (a << 21) | b, (v << 42) | (b << 21) | a]     # both directions
    keys = np.unique(np.concatenate(keys))      # once per visit
    a = (keys >> 21) & (2**21 - 1)
    b = keys & (2**21 - 1)
    C = sp.csr_matrix((np.ones(len(keys), dtype=np.float32), (a, b)), shape=(split.n_items, split.n_items))
    # Ties are common in a sample (many pairs were seen in one visit only): the product with more past visits
    # wins. The extra is below 1, so it never changes the order of two different counts.
    visits = np.bincount(h.drop_duplicates(["session", "item"])["item"], minlength=split.n_items)
    C.data += (visits / (visits.max() + 1))[C.indices].astype(np.float32)
    return prune(C, neighbours)


def prune(C, neighbours):
    """Keep the `neighbours` largest values of each row."""
    C = sp.csr_matrix(C)
    rows, cols, vals = [], [], []
    for r in range(C.shape[0]):
        a, b = C.indptr[r], C.indptr[r + 1]
        if a == b:
            continue
        best = np.lexsort((C.indices[a:b], -C.data[a:b]))[:neighbours]
        rows.append(np.full(len(best), r))
        cols.append(C.indices[a:b][best])
        vals.append(C.data[a:b][best])
    return sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=C.shape)


def covisit_scores(split, C):
    """Score each product by its co-visitation counts with the products of the visit, weighted by
    session_weights (recent events and carts count more)."""
    return session_matrix(split, session_weights(split)) @ C


C, seconds = timed(covisitation, test)
print(f"co-visitation: {C.nnz:,} pairs kept, in {seconds:.0f} s")
cov = top_k(covisit_scores(test, C))
lists["co-visitation"] = fill(cov, pop)
results["co-visitation"], _ = evaluate(test, lists["co-visitation"])
print(f"co-visitation  score {results['co-visitation']['score'][0]:.3f}  "
      f"next click on a new product {results['co-visitation']['clicks_new']:.3f}")

> **Check.** You should see a score of 0.109, and 0.181 for the next click on a product the visitor had not seen.

Co-visitation never suggests the visit's own products (a product is not its own neighbour). So it finds new products, but it misses the products the visitor comes back to.

## 6. The last item (Markov)

Run the next cell. It counts, in past visits, which product came **directly after** each product, as in [Sessions, sequences and LLMs, briefly](https://learning.nextia-ai.com/courses/recommendations/m07/sessions-sequences-and-llms-briefly/). A test visit gets the products that most often came after its **last** product.

In [ ]:
def transitions(split):
    """'What visitors looked at next': counts of product a followed directly by product b in a past visit."""
    h = split.history
    s, i = h["session"].to_numpy(), h["item"].to_numpy()
    same = (s[1:] == s[:-1]) & (i[1:] != i[:-1])
    return sp.csr_matrix((np.ones(same.sum(), dtype=np.float32), (i[:-1][same], i[1:][same])),
                         shape=(split.n_items, split.n_items))


def last_item(split):
    """The product of each visit's last visible event."""
    ends = np.cumsum(split.seen) - 1
    return split.visible["item"].to_numpy()[ends]


def markov_scores(split, T):
    """The next-product counts of the last product seen (a Markov model: only the current product matters)."""
    return T[last_item(split)]


T = transitions(test)
mar = top_k(markov_scores(test, T))
lists["last item (Markov)"] = fill(mar, pop)
results["last item (Markov)"], _ = evaluate(test, lists["last item (Markov)"])
print(f"Markov  score {results['last item (Markov)']['score'][0]:.3f}  "
      f"next click on a new product {results['last item (Markov)']['clicks_new']:.3f}")

> **Check.** You should see a score of 0.062 and 0.133 for the next click on a new product.

## 7. ALS

Run the next cell. ALS learns a vector for each product from past visits (each past visit is a row, like a customer), as in [Hidden tastes](https://learning.nextia-ai.com/courses/recommendations/m05/hidden-tastes/). It uses only the products that at least 5 past visits touched. A test visit's vector is the weighted sum of its products' vectors; the list is the 20 products whose vectors point most in the same direction. We use the settings that were best on the **validation** day (Friday): 128 factors and regularisation 1. This takes about a minute.

In [ ]:
def fit_als(split, factors=64, regularization=0.1, iterations=15, min_visits=5):
    """ALS on past visits (visits x products, confidence 1 + log of the weighted events), on the products that at
    least `min_visits` past visits touched. Returns the model and those products."""
    from implicit.als import AlternatingLeastSquares
    h = split.history
    visits = np.bincount(h.drop_duplicates(["session", "item"])["item"], minlength=split.n_items)
    keep = np.flatnonzero(visits >= min_visits)
    column = np.full(split.n_items, -1)
    column[keep] = np.arange(len(keep))
    h = h[column[h["item"].to_numpy()] >= 0]
    codes, sessions = pd.factorize(h["session"])
    w = TYPE_WEIGHT[h["type"].to_numpy()].astype(np.float32)
    X = sp.csr_matrix((w, (codes, column[h["item"].to_numpy()])), shape=(len(sessions), len(keep)))
    X.data = 1 + np.log(X.data)
    model = AlternatingLeastSquares(factors=factors, regularization=regularization, iterations=iterations,
                                    random_state=37, num_threads=1, calculate_training_loss=False)
    model.fit(X, show_progress=False)
    return model, keep


def als_lists(split, als):
    """Cosine similarity, as in item-cf: every product vector scaled to length 1. A visit's vector is the weighted
    sum of the vectors of the products it touched (recent events and carts count more). Then the 20 products whose
    vectors point most in the same direction. The visit's own products may come back. A visit with no known
    product gets an empty list (-1)."""
    model, keep = als
    Y = model.item_factors / np.maximum(np.linalg.norm(model.item_factors, axis=1, keepdims=True), 1e-9)
    W = session_matrix(split, session_weights(split))[:, keep].astype(np.float32)
    known = np.asarray(W.sum(axis=1)).ravel() > 0
    out = np.full((len(split), K), -1)
    for a in range(0, len(split), 1000):
        scores = (W[a:a + 1000] @ Y) @ Y.T
        best = np.argpartition(-scores, K, axis=1)[:, :K]
        order = np.argsort(-np.take_along_axis(scores, best, axis=1), axis=1, kind="stable")
        out[a:a + 1000] = keep[np.take_along_axis(best, order, axis=1)]
    out[~known] = -1
    return out


als, seconds = timed(fit_als, test, factors=128, regularization=1.0)
print(f"ALS on {len(als[1]):,} products, in {seconds:.0f} s")
al = als_lists(test, als)
lists["ALS"] = fill(al, pop)
results["ALS"], _ = evaluate(test, lists["ALS"])
print(f"ALS  score {results['ALS']['score'][0]:.3f}  next click on a new product {results['ALS']['clicks_new']:.3f}")

> **Check.** You should see a score of 0.275.

ALS can suggest the visit's own products, because a product's vector points in its own direction. That is why its score is higher than co-visitation's. For new products, it is weaker (0.071 against 0.181).

## 8. Combine with the visit, and compare

Each method is good at a different part: the visit's own products for carts and orders, the others for new products. So we put the visit's own products first, then fill the free places with each method.

> **Your turn.** Make the lists **visit + co-visitation**: the visit's own products (`session_items(test)`), then the co-visitation products (`cov`), then popular products. Use `combine(first, then)` and `fill(lists, pop)`. Store the lists in `design` and the score in `design_score` (`evaluate` returns the results first; the score is `["score"][0]`).

In [ ]:
design = fill(combine(session_items(test), cov), pop)
results["visit + co-visitation"], per_design = evaluate(test, design)
design_score = results["visit + co-visitation"]["score"][0]
print(round(design_score, 3))

In [ ]:
expect('design_score', rounded(design_score), 0.616, tolerance=0.001)

Run the next cell. It makes the same combination with Markov and with ALS, and shows every method in one table, on the same visits with the same measures.

In [ ]:
def table(results):
    """One row per method: recall@20 for clicks, carts and orders, the score (95% interval), MRR@20, coverage."""
    rows = []
    for name, o in results.items():
        rows.append({"method": name, "clicks": round(o["clicks"][0], 3), "carts": round(o["carts"][0], 3),
                     "orders": round(o["orders"][0], 3), "score": round(o["score"][0], 3),
                     "interval": f"{o['score'][1]:.3f} to {o['score'][2]:.3f}", "MRR@20": round(o["mrr"], 3),
                     "new next click": round(o["clicks_new"], 3), "coverage": f"{o['coverage']:.2%}"})
    return pd.DataFrame(rows).set_index("method")


lists["visit + co-visitation"] = fill(combine(session_items(test), cov), pop)
lists["visit + Markov"] = fill(combine(session_items(test), mar), pop)
lists["visit + ALS"] = fill(combine(session_items(test), al), pop)
for name in ("visit + co-visitation", "visit + Markov", "visit + ALS"):
    results[name], _ = evaluate(test, lists[name])
table(results)

> **Check.** You should see visit + co-visitation first, with a score of 0.616; visit + ALS 0.586; visit + Markov 0.602; the visit so far 0.568; popular 0.014.

Run the next cell. It shows the score by segment for four methods.

In [ ]:
shown = ["popular", "visit so far", "co-visitation", "visit + co-visitation"]
pd.DataFrame({name: {g: round(results[name]["segments"][g]["score"], 3) for g in SEGMENTS} for name in shown})

**What this tells us.** The longer the visit, the more its own products help. With 1 event seen, the visit so far has one product, and the co-visitation neighbours fill the other 19 places. In every segment, visit + co-visitation is first.

## 9. Failure case: a rule from another shop

Many shops never show a product the customer already bought. Suppose Jonas copies that rule as "never show a product the visitor already **saw**".

> **Predict.** What happens to the score, and to the next click on a new product?

Run the next cell. `without_seen` removes the visit's own products and fills the list with co-visitation and popular products the visitor has not seen.

In [ ]:
def without_seen(split, C, pop):
    """A rule copied from a shop with logged-in customers: never show a product the visitor has already seen.
    The list is then co-visitation (products not seen yet), filled with popular products not seen yet."""
    cov = top_k(covisit_scores(split, C), k=2 * K)
    out = np.full((len(split), K), -1)
    for r, seen in enumerate(split.seen_sets):
        row = [p for p in cov[r] if p >= 0 and p not in seen]
        row += [p for p in pop if p not in seen and p not in row]
        out[r] = row[:K]
    return out


rule, _ = evaluate(test, without_seen(test, C, pop))
d = results["visit + co-visitation"]
print(f"visit + co-visitation   score {d['score'][0]:.3f}  next click on a new product {d['clicks_new']:.3f}")
print(f"seen products removed   score {rule['score'][0]:.3f}  next click on a new product {rule['clicks_new']:.3f}")

> **Check.** You should see the score fall from 0.616 to 0.057.

**What this tells us.** In a visit, a product already seen is the best guess for a cart or an order. A rule is a decision about one page and one shop: measure it before you copy it.

## 10. End to end on new input: Sunday 4 September

Sunday was never used. Run the next cell. It does the whole design again one day later: co-visitation from every event before Sunday, then the lists for the visits that start on Sunday.

> **Check.** You should see 48,554 visits and a score of 0.613 for the design (popular 0.010, the visit so far 0.559).

In [ ]:
sunday = Split(events, day=6)
pop_sunday = popular(sunday)
C_sunday = covisitation(sunday)
final = fill(combine(session_items(sunday), top_k(covisit_scores(sunday, C_sunday))), pop_sunday)
check = {"popular": fill(np.full((len(sunday), K), -1), pop_sunday),
         "visit so far": fill(session_items(sunday), pop_sunday),
         "visit + co-visitation": final}
print(f"{len(sunday):,} visits start on {DAYS[6]}")
table({name: evaluate(sunday, l)[0] for name, l in check.items()})

A service does not make lists for a whole day at once. It gets one visit, at the moment the visitor opens a page. Run the next cell. `suggest_one` makes the same list for one visit, from its events so far. It prints one Sunday visit, its list, and what the visitor did next.

In [ ]:
def suggest_one(items, types, C, pop):
    """The design for one visit, as a service would run it: the visit's own products by weight, then their
    co-visitation neighbours, then popular products. items and types: the visit's events so far, in order."""
    n = len(items)
    recency = 0.1 + 0.9 * np.arange(n) / (n - 1) if n > 1 else np.ones(1)
    weights = TYPE_WEIGHT[types] * recency
    own = {}
    for i, w in zip(items.tolist(), weights):
        own[i] = own.get(i, 0.0) + w
    row = sorted(own, key=lambda i: (-own[i], i))[:K]
    if len(row) < K:
        v = sp.csr_matrix((list(own.values()), ([0] * len(own), list(own))), shape=(1, C.shape[0])) @ C
        for i in np.lexsort((v.indices, -v.data)):
            if v.indices[i] not in own:
                row.append(int(v.indices[i]))
                if len(row) == K:
                    break
    row += [p for p in pop if p not in row][:K - len(row)]
    return row


row = list(sunday.sessions).index(14328850)
end = np.cumsum(sunday.seen)[row]
visit = sunday.visible.iloc[end - sunday.seen[row]:end]
print("Seen so far:", ", ".join(f"{TYPES[t][:-1]} {aids[i]}" for i, t in zip(visit["item"], visit["type"])))
start = time.perf_counter()
suggestions = suggest_one(visit["item"].to_numpy(), visit["type"].to_numpy(), C_sunday, pop_sunday)
print(f"List made in {(time.perf_counter() - start) * 1000:.1f} ms")
for place, p in enumerate(suggestions, 1):
    marks = [m for m, hit in (("next click", p == sunday.next_click[row]),
                              ("carted", p in sunday.truth["carts"].get(row, set())),
                              ("ordered", p in sunday.truth["orders"].get(row, set()))) if hit]
    print(f"{place:2}. product {aids[p]:>8}  {'seen' if p in sunday.seen_sets[row] else '    '}  {', '.join(marks)}")

> **Check.** The list should be the same as the row of `final` for this visit: `suggestions == final[row].tolist()` is `True`. You should see 1 product(s) marked `carted` and 0 marked `ordered`.

## 11. Recap

You split real browsing sessions by day, cut each visit at a random point, and compared popularity, the visit's own products, co-visitation, the last item (Markov) and ALS, for clicks, carts and orders, by segment. In these visits, the products the visitor already saw are the best guess for carts and orders; co-visitation adds the best new products; popularity is far behind. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m10/next-item-from-browsing-sessions/#the-recommendation) turns this into a recommendation with its costs, its limits and what a person still checks.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Books for readers](https://learning.nextia-ai.com/courses/recommendations/m10/books-for-readers/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m10/next-item-from-browsing-sessions/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real: a sample of the OTTO Recommender Systems Dataset, CC BY 4.0. IDs are anonymised.